In [1]:
import os, sys, glob, time, warnings, collections

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.decomposition   import PCA
from sklearn.metrics         import (accuracy_score, f1_score,
                                     classification_report, confusion_matrix)
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.neural_network  import MLPClassifier

In [4]:
import torch, torch.nn as nn 
from torch.utils.data import DataLoader, TensorDataset

In [5]:
def load_one_file(path, short_threshold=None):
    """
    Load one QFlow lite .npy file.
 
    Parameters
    ----------
    path             : str    path to the .npy file
    short_threshold  : float  mean |current| (A) above which a D=0 file is called
                              Short Circuit. If None, label is left as -1 (D=0,
                              unresolved) — load_dataset resolves it after computing
                              the data-driven median threshold.
 
    Returns
    -------
    current  : (H, W) float32
    sensor0  : (H, W) float32
    sensor1  : (H, W) float32
    state_map: (H, W) int8
    label    : int   0=Barrier  1=Single Dot  2=Double Dot  3=Short Circuit
                     (-1 if D=0 and short_threshold is None)
    mean_cur : float  mean |current| (returned so load_dataset can threshold later)
    """
    data   = np.load(path, allow_pickle=True).item()
    output = data["output"]          # array of dicts, one per pixel
 
    n_px = len(output)
    side = int(round(n_px ** 0.5))  # infer grid side from data (100 for QFlow lite)
 
    current   = np.array([o["current"]   for o in output], dtype=np.float32)
    sensor0   = np.array([o["sensor"][0] for o in output], dtype=np.float32)
    sensor1   = np.array([o["sensor"][1] for o in output], dtype=np.float32)
    state_arr = np.array([o["state"]     for o in output], dtype=np.int8)
 
    current   = current.reshape(side, side)
    sensor0   = sensor0.reshape(side, side)
    sensor1   = sensor1.reshape(side, side)
    state_map = state_arr.reshape(side, side)
 
    max_state = int(state_arr.max())
    mean_cur  = float(np.abs(current).mean())
 
    if max_state == 1:
        label = 1
    elif max_state == 2:
        label = 2
    else:                            # max_state == 0: barrier or short circuit
        if short_threshold is None:
            label = -1               # deferred: resolved in load_dataset
        else:
            label = 3 if mean_cur > short_threshold else 0
 
    return current, sensor0, sensor1, state_map, label, mean_cur

In [6]:
def load_dataset(data_dir, class_names=None):
    """
    Load every .npy file in data_dir.
 
    The Barrier / Short-Circuit threshold is determined automatically as the
    median mean-|current| across all D=0 (state-map-all-zero) files, so the
    two classes are always balanced regardless of the physical current scale.
 
    Parameters
    ----------
    data_dir    : str   folder containing .npy files
    class_names : list  used only for the printed summary
 
    Returns
    -------
    X_cur  : (N, H*W) float32
    X_sen0 : (N, H*W) float32
    X_sen1 : (N, H*W) float32
    X_smap : (N, H*W) float32
    y      : (N,)     int64
    paths  : list[str]
    """
    if class_names is None:
        class_names = ["Barrier", "Single Dot", "Double Dot", "Short Circuit"]
 
    npy_files = sorted(glob.glob(os.path.join(data_dir, "*.npy")))
    if not npy_files:
        print(f"  [!] No .npy files found in '{data_dir}'.")
        return None
 
    print(f"  Found {len(npy_files)} .npy files – loading ...")
    t0 = time.time()
 
    currents, sens0, sens1, smaps, labels, mean_curs = [], [], [], [], [], []
    failed = 0
 
    for i, path in enumerate(npy_files):
        try:
            cur, s0, s1, sm, lbl, mc = load_one_file(path, short_threshold=None)
            currents.append(cur.ravel())
            sens0.append(s0.ravel())
            sens1.append(s1.ravel())
            smaps.append(sm.ravel())
            labels.append(lbl)
            mean_curs.append(mc)
        except Exception as e:
            failed += 1
            print(f"    skip {os.path.basename(path)}: {e}")
 
        if (i + 1) % 200 == 0:
            print(f"    {i + 1}/{len(npy_files)} ...")
 
    # ── Resolve Barrier vs Short Circuit using a data-driven threshold ──────
    # Use the median mean-|current| of D=0 files so the split is always ~50/50
    labels    = np.array(labels,    dtype=np.int64)
    mean_curs = np.array(mean_curs, dtype=np.float64)
    d0_mask   = labels == -1
    if d0_mask.sum() > 0:
        threshold = float(np.median(mean_curs[d0_mask]))
        print(f"  Auto threshold (Barrier/Short Circuit): {threshold:.3e} A")
        labels[d0_mask] = np.where(mean_curs[d0_mask] > threshold, 3, 0)
    else:
        labels = labels  # no deferred labels
 
    N = int(d0_mask.shape[0])
    print(f"  Loaded {N} samples in {time.time() - t0:.1f}s  ({failed} failed)")
    dist = collections.Counter(labels.tolist())
    print(f"  Class distribution: { {class_names[k]: v for k, v in sorted(dist.items())} }")
 
    return (
        np.array(currents, dtype=np.float32),
        np.array(sens0,    dtype=np.float32),
        np.array(sens1,    dtype=np.float32),
        np.array(smaps,    dtype=np.float32),
        labels,
        npy_files[:N],
    )

In [7]:
dat = load_dataset("C:/Users/Lehma/AIProjects/data_qflow_lite")

  Found 1001 .npy files – loading ...
    200/1001 ...
    400/1001 ...
    600/1001 ...
    800/1001 ...
    1000/1001 ...
  Loaded 1001 samples in 174.6s  (0 failed)
  Class distribution: {'Double Dot': 1001}


In [8]:
CFG = {
    "data_dir"       : "C:/Users/Lehma/AIProjects/data_qflow_lite",
    "results_dir"    : "C:/Users/Lehma/AIProjects/data_qflow_lite",
    "seed"           : 42,
    "img_size"       : 100,
    "short_threshold": 1e-5,
    "pca_components" : 100,
    "cv_folds"       : 5,
    "test_size"      : 0.20,
    "class_names"    : ["Barrier", "Single Dot", "Double Dot", "Short Circuit"],
    "class_colors"   : ["#4C72B0", "#55A868", "#C44E52", "#8172B2"],
}
 
np.random.seed(CFG["seed"])
os.makedirs(CFG["results_dir"], exist_ok=True)

In [10]:
def plot_samples(X_cur, X_s0, X_s1, X_sm, y,
                 class_names, results_dir, img_size=100, n_per_class=3):
    n_cols = n_per_class * 3
    n_rows = len(class_names)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_per_class * 7, 11),
                             constrained_layout=True)
    fig.suptitle("QFlow Lite – Sample Maps per Device State", fontsize=14)
 
    cmaps      = ["viridis", "plasma", "tab10"]
    col_titles = ["Current", "Sensor 0", "State Map"]
 
    for cls_idx, cls_name in enumerate(class_names):
        idxs = np.where(y == cls_idx)[0][:n_per_class]
        for j, si in enumerate(idxs):
            maps = [
                X_cur[si].reshape(img_size, img_size),
                X_s0[si].reshape(img_size, img_size),
                X_sm[si].reshape(img_size, img_size),
            ]
            for k, (mdata, cmap) in enumerate(zip(maps, cmaps)):
                ax = axes[cls_idx, j * 3 + k]
                ax.imshow(mdata, cmap=cmap, origin="lower", aspect="auto")
                ax.set_xticks([]); ax.set_yticks([])
                if j == 0 and k == 0:
                    ax.set_ylabel(cls_name, fontsize=11, fontweight="bold")
                if cls_idx == 0 and j == 0:
                    ax.set_title(col_titles[k], fontsize=9)
    plt.show()
    plt.close()

In [11]:
def build_feature_matrix(X_cur, X_s0, X_s1, X_sm):
    """Concatenate all four flat maps -> (N, 4 * H * W)."""
    return np.concatenate([X_cur, X_s0, X_s1, X_sm], axis=1)

In [12]:
def build_pipelines(n_pca=100, seed=42):
    pca_step = [("pca", PCA(n_components=n_pca, random_state=seed))]
    return {
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", RandomForestClassifier(
                n_estimators=500, max_features="sqrt",
                n_jobs=-1, random_state=seed)),
        ]),
        "SVM (RBF)": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", SVC(kernel="rbf", C=10, gamma="scale",
                        class_weight="balanced", random_state=seed)),
        ]),
        "Gradient Boosting": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", GradientBoostingClassifier(
                n_estimators=300, max_depth=4,
                learning_rate=0.05, subsample=0.8,
                random_state=seed)),
        ]),
        "k-NN (k=7)": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", KNeighborsClassifier(n_neighbors=7, n_jobs=-1)),
        ]),
        "MLP": Pipeline([
            ("scaler", StandardScaler()), *pca_step,
            ("clf", MLPClassifier(
                hidden_layer_sizes=(512, 256, 128, 64),
                activation="relu", solver="adam",
                batch_size=32, max_iter=500,
                early_stopping=True, validation_fraction=0.1,
                random_state=seed)),
        ]),
    }

In [13]:
def evaluate_models(X, y, pipes, cv_folds=5, test_size=0.20, seed=42):
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed)
 
    results = []
    for name, pipe in pipes.items():
        print(f"\n  +-- {name}")
        t0 = time.time()
        cv_scores = cross_val_score(pipe, X_tr, y_tr,
                                    cv=skf, scoring="accuracy", n_jobs=-1)
        print(f"  |   CV  Acc : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}"
              f"  ({time.time() - t0:.1f}s)")
        pipe.fit(X_tr, y_tr)
        y_pred   = pipe.predict(X_te)
        test_acc = accuracy_score(y_te, y_pred)
        test_f1  = f1_score(y_te, y_pred, average="weighted")
        print(f"  |   Test Acc: {test_acc:.4f}   Weighted F1: {test_f1:.4f}")
 
        results.append({
            "model"   : name,
            "cv_mean" : cv_scores.mean(),
            "cv_std"  : cv_scores.std(),
            "test_acc": test_acc,
            "test_f1" : test_f1,
            "y_true"  : y_te,
            "y_pred"  : y_pred,
        })
    return results, y_te

In [14]:
def train_cnn(X_cur, X_s0, X_s1, X_sm, y,
              class_names, results_dir,
              img_size=100, test_size=0.20,
              epochs=30, batch_size=32, lr=1e-3, seed=42):
    try:
        import torch, torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset
    except ImportError:
        print("  PyTorch not installed – skipping CNN.")
        return None
 
    n_classes = len(class_names)
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n-- CNN  (device={device}) ---")
 
    X_img = np.stack([
        X_cur.reshape(-1, img_size, img_size),
        X_s0.reshape(-1,  img_size, img_size),
        X_s1.reshape(-1,  img_size, img_size),
        X_sm.reshape(-1,  img_size, img_size),
    ], axis=1).astype(np.float32)
 
    for c in range(4):
        mu = X_img[:, c].mean(); sig = X_img[:, c].std() + 1e-8
        X_img[:, c] = (X_img[:, c] - mu) / sig
 
    n    = len(y)
    idx  = np.random.permutation(n)
    n_te = int(n * test_size); n_va = int(n * 0.10)
    tr_idx = idx[n_te + n_va:]; va_idx = idx[n_te:n_te + n_va]; te_idx = idx[:n_te]
 
    def make_loader(idxs, shuffle=True):
        return DataLoader(
            TensorDataset(torch.tensor(X_img[idxs]),
                          torch.tensor(y[idxs], dtype=torch.long)),
            batch_size=batch_size, shuffle=shuffle)
 
    tr_ldr = make_loader(tr_idx); va_ldr = make_loader(va_idx, False)
    te_ldr = make_loader(te_idx, False)
 
    class QDotCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Conv2d(4,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
                nn.Conv2d(32,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
                nn.Conv2d(64,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
                nn.Conv2d(128,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
                nn.MaxPool2d(2), nn.AdaptiveAvgPool2d(4),
            )
            self.head = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128*16, 256), nn.ReLU(), nn.Dropout(0.4),
                nn.Linear(256, 64),     nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(64, n_classes),
            )
        def forward(self, x): return self.head(self.enc(x))
 
    model     = QDotCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
 
    tr_accs, va_accs = [], []
    best_va = 0.0; best_state = None
 
    for epoch in range(1, epochs + 1):
        model.train(); c = t = 0
        for Xb, yb in tr_ldr:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb); loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            c += (out.argmax(1) == yb).sum().item(); t += len(yb)
        tr_accs.append(c / t)
 
        model.eval(); c = t = 0
        with torch.no_grad():
            for Xb, yb in va_ldr:
                Xb, yb = Xb.to(device), yb.to(device)
                c += (model(Xb).argmax(1) == yb).sum().item(); t += len(yb)
        va_acc = c / t; va_accs.append(va_acc); scheduler.step()
 
        if va_acc > best_va:
            best_va = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs}  train={tr_accs[-1]:.4f}  val={va_acc:.4f}")
 
    model.load_state_dict(best_state); model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for Xb, yb in te_ldr:
            preds.extend(model(Xb.to(device)).argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
 
    test_acc = accuracy_score(trues, preds)
    test_f1  = f1_score(trues, preds, average="weighted")
    print(f"\n  CNN Test Acc: {test_acc:.4f}   Weighted F1: {test_f1:.4f}")
    print(classification_report(trues, preds, target_names=class_names))
 
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(tr_accs, lw=2, label="Train")
    ax.plot(va_accs, lw=2, ls="--", label="Validation")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.set_title("CNN Learning Curves"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(results_dir, "cnn_learning_curves.png")
    plt.savefig(path, dpi=120, bbox_inches="tight"); plt.close()
    print(f"  saved: {path}")
 
    return {"model": "CNN (PyTorch)", "cv_mean": None, "cv_std": None,
            "test_acc": test_acc, "test_f1": test_f1,
            "y_true": np.array(trues), "y_pred": np.array(preds)}